# 04 — Controlled Follow-Up Question Selection

## Purpose

This notebook implements controlled follow-up question selection for claims routed to `INFO_REQUIRED`.

The system will not generate arbitrary customer questions. It will select from the approved synthetic follow-up-question catalogue using deterministic mappings from the triggered rule and observed missing-information or evidence condition.

## Safety boundaries

- Follow-up questions are selected only for `INFO_REQUIRED` triage outcomes.
- Questions come only from the approved catalogue.
- The selector does not change the deterministic triage outcome, rule, or precedence.
- Missing or unmapped catalogue entries are surfaced for analyst review rather than invented by an LLM.
- Held-out evaluation data is not loaded or used in this notebook.

## Step 1 — Inspect the Follow-Up Question Catalogue

Before building the selector, inspect the actual runtime dataset keys and the catalogue schema. This ensures the component uses the approved source fields rather than assumed column names.

In [1]:
# ============================================================
# Step 1: Follow-Up Question Catalogue Inspection
# ============================================================

from pathlib import Path
import sys

import pandas as pd


CURRENT_DIRECTORY = Path.cwd().resolve()

if (CURRENT_DIRECTORY / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY
elif (CURRENT_DIRECTORY.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_runtime_data

data = load_runtime_data()

print("Available runtime data keys:")
print(sorted(data.keys()))

follow_up_keys = [
    key
    for key in data
    if "follow" in key.casefold()
    or "question" in key.casefold()
]

print("\nPossible follow-up catalogue keys:")
print(follow_up_keys)

for key in follow_up_keys:
    dataset = data[key]

    if isinstance(dataset, pd.DataFrame):
        print(f"\n{'=' * 70}")
        print(f"Dataset key: {key}")
        print("Shape:", dataset.shape)
        print("Columns:")
        print(dataset.columns.tolist())
        print("\nSample rows:")
        display(dataset.head(10))

Available runtime data keys:
['coverage_matrix', 'development_claims', 'evidence_bundles', 'evidence_documents', 'evidence_requirements', 'follow_up_question_catalog', 'follow_up_question_selection_rules', 'plan_master', 'policy_lookup', 'policy_rules', 'prior_claims', 'risk_results']

Possible follow-up catalogue keys:
['follow_up_question_catalog', 'follow_up_question_selection_rules']

Dataset key: follow_up_question_catalog
Shape: (14, 16)
Columns:
['question_id', 'question_name', 'trigger_type', 'trigger_condition', 'source_rule_id', 'decision_stage', 'applicable_claim_categories', 'evidence_profile_id', 'response_field', 'response_format', 'question_priority', 'customer_facing_question', 'plain_language_reason', 'agent_handling_after_response', 'max_repeats', 'active_flag']

Sample rows:


,question_id,question_name,trigger_type,trigger_condition,source_rule_id,decision_stage,applicable_claim_categories,evidence_profile_id,response_field,response_format,question_priority,customer_facing_question,plain_language_reason,agent_handling_after_response,max_repeats,active_flag
0,FUP-001,Confirm policy or customer reference,MISSING_FIELD,No unique policy eligibility record can be fou...,DATA-001,ELIGIBILITY_FACTS,ALL,NaN,policy_or_customer_reference,TEXT,10,Please provide your policy reference or custom...,This helps us identify the protection plan lin...,Re-run policy lookup. If multiple authoritativ...,1,Yes
1,FUP-002,Confirm incident date,MISSING_OR_INVALID_FIELD,"Reported incident date is missing, ambiguous, ...",ELG-001,ELIGIBILITY_FACTS,ALL,NaN,reported_incident_date,DATE_YYYY_MM_DD,20,Please confirm the date on which the incident ...,We need the incident date to continue assessin...,Validate the date and re-run policy-active-on-...,1,Yes
2,FUP-003,Provide claimed device identifier,MISSING_OR_INVALID_FIELD,"Claimed device identifier is missing, malforme...",DEV-001,ELIGIBILITY_FACTS,ALL,NaN,claimed_device_identifier,TEXT,30,Please provide the IMEI or serial number for t...,This helps us confirm the covered-device details.,Re-run device match. An unresolved mismatch is...,1,Yes
3,FUP-004,Clarify incident category,MISSING_OR_AMBIGUOUS_FIELD,Claim category is UNSPECIFIED or cannot be rel...,CLM-001,ELIGIBILITY_FACTS,UNSPECIFIED,NaN,claim_category_selected,ENUM_OR_FREE_TEXT,40,Please describe what happened to the device an...,We need to understand the type of incident bef...,Classify only when the response is sufficientl...,1,Yes
4,FUP-005,Upload screen-damage photo,MISSING_REQUIRED_EVIDENCE,Required DAMAGE_PHOTO is absent for SCREEN_DAM...,EVD-001,REQUIRED_EVIDENCE,SCREEN_DAMAGE,EVD-SCREEN-01,DAMAGE_PHOTO,FILE_UPLOAD,50,Please upload a clear photo showing the report...,The image helps us confirm the visible screen ...,"Check for a readable, relevant photo. A repair...",1,Yes
5,FUP-006,Re-upload readable screen-damage photo,UNREADABLE_OR_INVALID_EVIDENCE,Submitted DAMAGE_PHOTO for SCREEN_DAMAGE is ma...,EVD-001,REQUIRED_EVIDENCE,SCREEN_DAMAGE,EVD-SCREEN-01,DAMAGE_PHOTO,FILE_UPLOAD,50,The uploaded image could not be read clearly. ...,We need a readable image to continue assessing...,If the replacement upload materially conflicts...,1,Yes
6,FUP-007,Upload accidental-damage photo,MISSING_REQUIRED_EVIDENCE,Required DAMAGE_PHOTO is absent for ACCIDENTAL...,EVD-001,REQUIRED_EVIDENCE,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,DAMAGE_PHOTO,FILE_UPLOAD,50,Please upload a clear photo showing the report...,The image helps us confirm the visible acciden...,"Check for a readable, relevant photo. A repair...",1,Yes
7,FUP-008,Re-upload readable accidental-damage photo,UNREADABLE_OR_INVALID_EVIDENCE,Submitted DAMAGE_PHOTO for ACCIDENTAL_DAMAGE i...,EVD-001,REQUIRED_EVIDENCE,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,DAMAGE_PHOTO,FILE_UPLOAD,50,The uploaded image could not be read clearly. ...,We need a readable image to continue assessing...,If the replacement upload materially conflicts...,1,Yes
8,FUP-009,Upload liquid-damage photo,MISSING_REQUIRED_EVIDENCE,Required DAMAGE_PHOTO is absent for LIQUID_DAM...,EVD-001,REQUIRED_EVIDENCE,LIQUID_DAMAGE,EVD-LIQUID-01,DAMAGE_PHOTO,FILE_UPLOAD,50,Please upload a clear photo showing the report...,The image helps us assess the reported liquid-...,"Check for a readable, relevant photo. A repair...",1,Yes
9,FUP-010,Re-upload readable liquid-damage photo,UNREADABLE_OR_INVALID_EVIDENCE,Submitted DAMAGE_PHOTO for LIQUID_DAMAGE is ma...,EVD-001,REQUIRED_EVIDENCE,LIQUID_DAMAGE,EVD-LIQUID-01,DAMAGE_PHOTO,FILE_UPLOAD,50,The uploaded image could not be read clearly. ...,We need a readable image to continue assessing...,If the replacement upload materially conflicts...,1,Yes



Dataset key: follow_up_question_selection_rules
Shape: (7, 6)
Columns:
['selection_rule_id', 'rule_name', 'priority', 'condition', 'agent_action', 'rationale']

Sample rows:


,selection_rule_id,rule_name,priority,condition,agent_action,rationale
0,FSEL-001,Higher-precedence outcome blocks follow-up,10,Any applicable rule has already produced MANUA...,Do not ask follow-up questions. Return the exi...,The agent must not burden the customer with do...
1,FSEL-002,Follow-up only for INFO_REQUIRED,20,An applicable deterministic rule produces INFO...,Select only catalogue questions mapped to the ...,Questions must be rule-grounded and not improv...
2,FSEL-003,Eligibility facts before evidence,30,Both eligibility facts and required evidence a...,Ask eligibility-fact questions first. Reassess...,The system should not request evidence for a c...
3,FSEL-004,Maximum questions per interaction,40,More than one valid catalogue question is sele...,Ask a maximum of three questions in one messag...,Reduces customer burden and keeps the interact...
4,FSEL-005,No duplicate question,50,The same response field has already been reque...,Use the same question at most once. If still u...,Prevents looping and supports bounded agent be...
5,FSEL-006,No customer resolution of internal conflicts,60,"The issue is a source-system conflict, duplica...",Do not generate a customer question. Route to ...,Customers must not be asked to solve internal ...
6,FSEL-007,Neutral communication,70,A follow-up message is generated.,Use the catalogue wording with a neutral expla...,Matches the operational safety and communicati...


## Step 1B — Locate the Approved Follow-Up Catalogue

The runtime loader currently exposes no follow-up-question dataset.

This step searches the active project data folders, excluding held-out evaluation assets, to determine whether the approved catalogue exists on disk but has not yet been added to the runtime loader.

In [2]:
# ============================================================
# Step 1B: Locate the Follow-Up Question Catalogue
# ============================================================

from IPython.display import display

SEARCH_TERMS = (
    "follow",
    "question",
    "template",
    "information",
    "evidence",
)

data_root = PROJECT_ROOT / "data"

file_rows = []

for file_path in sorted(data_root.rglob("*")):
    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(PROJECT_ROOT)

    # Never inspect held-out evaluation assets in this notebook.
    if "evaluation" in relative_path.parts:
        continue

    if file_path.suffix.casefold() not in {".csv", ".json", ".md"}:
        continue

    filename = file_path.name.casefold()

    name_match = any(
        term in filename
        for term in SEARCH_TERMS
    )

    content_match = False

    try:
        preview_text = file_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )[:3000].casefold()

        content_match = any(
            term in preview_text
            for term in SEARCH_TERMS
        )
    except OSError:
        preview_text = ""

    file_rows.append(
        {
            "relative_path": str(relative_path),
            "file_type": file_path.suffix.casefold(),
            "size_bytes": file_path.stat().st_size,
            "name_match": name_match,
            "content_match": content_match,
        }
    )

active_file_inventory = pd.DataFrame(file_rows)

print("Active non-evaluation data files found:", len(active_file_inventory))

catalogue_candidates = active_file_inventory[
    active_file_inventory["name_match"]
    | active_file_inventory["content_match"]
].copy()

print("\nPossible follow-up catalogue candidates:")
display(catalogue_candidates)

print("\nFull active data-file inventory:")
display(active_file_inventory)

Active non-evaluation data files found: 113

Possible follow-up catalogue candidates:


,relative_path,file_type,size_bytes,name_match,content_match
0,data/knowledge_base/KB-001_claims_triage_sop_v...,.md,3480,False,True
1,data/knowledge_base/KB-002_evidence_review_gui...,.md,3167,True,True
2,data/knowledge_base/KB-003_follow_up_communica...,.md,2813,True,True
3,data/knowledge_base/KB-004_manual_review_escal...,.md,2590,False,True
4,data/knowledge_base/KB-005_decision_explanatio...,.md,2261,False,True
...,...,...,...,...,...
104,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.md,3407,True,True
105,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.md,2956,False,True
109,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.md,1375,False,True
110,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.md,2421,False,True



Full active data-file inventory:


,relative_path,file_type,size_bytes,name_match,content_match
0,data/knowledge_base/KB-001_claims_triage_sop_v...,.md,3480,False,True
1,data/knowledge_base/KB-002_evidence_review_gui...,.md,3167,True,True
2,data/knowledge_base/KB-003_follow_up_communica...,.md,2813,True,True
3,data/knowledge_base/KB-004_manual_review_escal...,.md,2590,False,True
4,data/knowledge_base/KB-005_decision_explanatio...,.md,2261,False,True
...,...,...,...,...,...
108,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.json,1142,False,False
109,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.md,1375,False,True
110,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.md,2421,False,True
111,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.csv,910,False,True


## Step 1C — Identify the Structured Follow-Up Catalogue Source

The previous file search identified follow-up-related documentation and staging assets.

This step filters only filenames that explicitly reference follow-up questions or templates, then previews any structured CSV or JSON candidates. The objective is to locate the approved catalogue rather than infer or create questions.

In [3]:
# ============================================================
# Step 1C: Identify Structured Follow-Up Catalogue Source
# ============================================================

import json
import re

FILENAME_PATTERN = re.compile(
    r"(follow.?up|follow_up|question|template)",
    flags=re.IGNORECASE,
)

filename_candidates = active_file_inventory[
    active_file_inventory["relative_path"].apply(
        lambda path: bool(FILENAME_PATTERN.search(Path(path).name))
    )
].copy()

print("Filename-based candidates:")
for relative_path in filename_candidates["relative_path"].tolist():
    print("-", relative_path)

structured_candidates = filename_candidates[
    filename_candidates["file_type"].isin({".csv", ".json"})
].copy()

print("\nStructured CSV / JSON candidates:")
display(structured_candidates)


def preview_json_candidate(payload: object) -> None:
    """Safely inspect a JSON file without assuming tabular structure."""
    print("Top-level JSON type:", type(payload).__name__)

    if isinstance(payload, list):
        print("Top-level list length:", len(payload))

        if payload and all(isinstance(item, dict) for item in payload):
            candidate_data = pd.json_normalize(payload)

            print("Detected tabular list-of-records structure.")
            print("Shape:", candidate_data.shape)
            print("Columns:", candidate_data.columns.tolist())

            display(candidate_data.head(10))
        else:
            print("Preview:")
            print(
                json.dumps(
                    payload[:3],
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                )[:4000]
            )

    elif isinstance(payload, dict):
        print("Top-level keys:", list(payload.keys()))

        nested_record_keys = [
            key
            for key, value in payload.items()
            if isinstance(value, list)
            and value
            and all(isinstance(item, dict) for item in value)
        ]

        if nested_record_keys:
            print(
                "Nested list-of-records keys found:",
                nested_record_keys,
            )

            for key in nested_record_keys:
                candidate_data = pd.json_normalize(payload[key])

                print(f"\nNested key: {key}")
                print("Shape:", candidate_data.shape)
                print("Columns:", candidate_data.columns.tolist())

                display(candidate_data.head(10))
        else:
            print("JSON preview:")
            print(
                json.dumps(
                    payload,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                )[:4000]
            )

    else:
        print("JSON preview:")
        print(str(payload)[:4000])


for relative_path in structured_candidates["relative_path"].tolist():
    file_path = PROJECT_ROOT / relative_path

    print("\n" + "=" * 80)
    print("Candidate:", relative_path)
    print("=" * 80)

    if file_path.suffix.casefold() == ".csv":
        candidate_data = pd.read_csv(file_path)

        print("Shape:", candidate_data.shape)
        print("Columns:", candidate_data.columns.tolist())

        display(candidate_data.head(10))

    elif file_path.suffix.casefold() == ".json":
        try:
            with file_path.open(
                "r",
                encoding="utf-8",
            ) as json_file:
                json_payload = json.load(json_file)

            preview_json_candidate(json_payload)

        except json.JSONDecodeError as error:
            print("Could not parse JSON:", error)

Filename-based candidates:
- data/knowledge_base/KB-003_follow_up_communication_guide_v1.md
- data/runtime/reference/follow_up_question_catalog_v1.csv
- data/runtime/reference/follow_up_question_data_dictionary_v1.csv
- data/runtime/reference/follow_up_question_selection_rules_v1.csv
- data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/data/knowledge_base/KB-003_follow_up_communication_guide_v1.md
- data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/data/reference/follow_up_question_catalog_v1.csv
- data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/data/reference/follow_up_question_data_dictionary_v1.csv
- data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/data/reference/follow_up_question_selection_rules_v1.csv
- data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/docs/follow_up_question_tool_contract_v1.json
- data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/docs/item_9_follow_up_question_catalog_v1.md

Structured CSV / JSON candidates:


,relative_path,file_type,size_bytes,name_match,content_match
20,data/runtime/reference/follow_up_question_cata...,.csv,8561,True,True
21,data/runtime/reference/follow_up_question_data...,.csv,2623,True,True
22,data/runtime/reference/follow_up_question_sele...,.csv,2320,True,True
69,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.csv,8561,True,True
70,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.csv,2623,True,True
71,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.csv,2320,True,True
93,data/staging/BYOC_DeviceProtect_Claims_Triage_...,.json,1183,True,True



Candidate: data/runtime/reference/follow_up_question_catalog_v1.csv
Shape: (14, 16)
Columns: ['question_id', 'question_name', 'trigger_type', 'trigger_condition', 'source_rule_id', 'decision_stage', 'applicable_claim_categories', 'evidence_profile_id', 'response_field', 'response_format', 'question_priority', 'customer_facing_question', 'plain_language_reason', 'agent_handling_after_response', 'max_repeats', 'active_flag']


,question_id,question_name,trigger_type,trigger_condition,source_rule_id,decision_stage,applicable_claim_categories,evidence_profile_id,response_field,response_format,question_priority,customer_facing_question,plain_language_reason,agent_handling_after_response,max_repeats,active_flag
0,FUP-001,Confirm policy or customer reference,MISSING_FIELD,No unique policy eligibility record can be fou...,DATA-001,ELIGIBILITY_FACTS,ALL,NaN,policy_or_customer_reference,TEXT,10,Please provide your policy reference or custom...,This helps us identify the protection plan lin...,Re-run policy lookup. If multiple authoritativ...,1,Yes
1,FUP-002,Confirm incident date,MISSING_OR_INVALID_FIELD,"Reported incident date is missing, ambiguous, ...",ELG-001,ELIGIBILITY_FACTS,ALL,NaN,reported_incident_date,DATE_YYYY_MM_DD,20,Please confirm the date on which the incident ...,We need the incident date to continue assessin...,Validate the date and re-run policy-active-on-...,1,Yes
2,FUP-003,Provide claimed device identifier,MISSING_OR_INVALID_FIELD,"Claimed device identifier is missing, malforme...",DEV-001,ELIGIBILITY_FACTS,ALL,NaN,claimed_device_identifier,TEXT,30,Please provide the IMEI or serial number for t...,This helps us confirm the covered-device details.,Re-run device match. An unresolved mismatch is...,1,Yes
3,FUP-004,Clarify incident category,MISSING_OR_AMBIGUOUS_FIELD,Claim category is UNSPECIFIED or cannot be rel...,CLM-001,ELIGIBILITY_FACTS,UNSPECIFIED,NaN,claim_category_selected,ENUM_OR_FREE_TEXT,40,Please describe what happened to the device an...,We need to understand the type of incident bef...,Classify only when the response is sufficientl...,1,Yes
4,FUP-005,Upload screen-damage photo,MISSING_REQUIRED_EVIDENCE,Required DAMAGE_PHOTO is absent for SCREEN_DAM...,EVD-001,REQUIRED_EVIDENCE,SCREEN_DAMAGE,EVD-SCREEN-01,DAMAGE_PHOTO,FILE_UPLOAD,50,Please upload a clear photo showing the report...,The image helps us confirm the visible screen ...,"Check for a readable, relevant photo. A repair...",1,Yes
5,FUP-006,Re-upload readable screen-damage photo,UNREADABLE_OR_INVALID_EVIDENCE,Submitted DAMAGE_PHOTO for SCREEN_DAMAGE is ma...,EVD-001,REQUIRED_EVIDENCE,SCREEN_DAMAGE,EVD-SCREEN-01,DAMAGE_PHOTO,FILE_UPLOAD,50,The uploaded image could not be read clearly. ...,We need a readable image to continue assessing...,If the replacement upload materially conflicts...,1,Yes
6,FUP-007,Upload accidental-damage photo,MISSING_REQUIRED_EVIDENCE,Required DAMAGE_PHOTO is absent for ACCIDENTAL...,EVD-001,REQUIRED_EVIDENCE,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,DAMAGE_PHOTO,FILE_UPLOAD,50,Please upload a clear photo showing the report...,The image helps us confirm the visible acciden...,"Check for a readable, relevant photo. A repair...",1,Yes
7,FUP-008,Re-upload readable accidental-damage photo,UNREADABLE_OR_INVALID_EVIDENCE,Submitted DAMAGE_PHOTO for ACCIDENTAL_DAMAGE i...,EVD-001,REQUIRED_EVIDENCE,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,DAMAGE_PHOTO,FILE_UPLOAD,50,The uploaded image could not be read clearly. ...,We need a readable image to continue assessing...,If the replacement upload materially conflicts...,1,Yes
8,FUP-009,Upload liquid-damage photo,MISSING_REQUIRED_EVIDENCE,Required DAMAGE_PHOTO is absent for LIQUID_DAM...,EVD-001,REQUIRED_EVIDENCE,LIQUID_DAMAGE,EVD-LIQUID-01,DAMAGE_PHOTO,FILE_UPLOAD,50,Please upload a clear photo showing the report...,The image helps us assess the reported liquid-...,"Check for a readable, relevant photo. A repair...",1,Yes
9,FUP-010,Re-upload readable liquid-damage photo,UNREADABLE_OR_INVALID_EVIDENCE,Submitted DAMAGE_PHOTO for LIQUID_DAMAGE is ma...,EVD-001,REQUIRED_EVIDENCE,LIQUID_DAMAGE,EVD-LIQUID-01,DAMAGE_PHOTO,FILE_UPLOAD,50,The uploaded image could not be read clearly. ...,We need a readable image to continue assessing...,If the replacement upload materially conflicts...,1,Yes



Candidate: data/runtime/reference/follow_up_question_data_dictionary_v1.csv
Shape: (16, 6)
Columns: ['field_name', 'data_type', 'nullable', 'business_definition', 'validation_or_usage_notes', 'example_value']


...


Candidate: data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/data/reference/follow_up_question_data_dictionary_v1.csv
Shape: (16, 6)
Columns: ['field_name', 'data_type', 'nullable', 'business_definition', 'validation_or_usage_notes', 'example_value']


,field_name,data_type,nullable,business_definition,validation_or_usage_notes,example_value
0,question_id,string,No,Unique controlled follow-up question identifier.,Pattern FUP-###; immutable once referenced in ...,FUP-001
1,question_name,string,No,Short operational label for the question.,Not shown to customers by default.,Confirm incident date
2,trigger_type,enum,No,Type of unmet information condition that activ...,MISSING_FIELD | MISSING_OR_INVALID_FIELD | MIS...,MISSING_REQUIRED_EVIDENCE
3,trigger_condition,string,No,Specific condition under which the question ma...,Must align with an INFO_REQUIRED condition; no...,Required DAMAGE_PHOTO is absent.
4,source_rule_id,string,No,Item #3 rule that grounds the question.,Must be an active policy-rule identifier.,EVD-001
5,decision_stage,enum,No,Decision stage in which the question may be as...,ELIGIBILITY_FACTS | REQUIRED_EVIDENCE.,ELIGIBILITY_FACTS
6,applicable_claim_categories,string,No,Supported claim categories to which the questi...,ALL or a valid Item #2 / Item #6 category.,THEFT
7,evidence_profile_id,string,No,Item #7 evidence profile connected to the ques...,N/A for non-evidence questions.,EVD-THEFT-01
8,response_field,string,No,Field or document type requested from the cust...,Used to prevent duplicate requests and track c...,reported_incident_date
9,response_format,enum,No,Expected response format.,TEXT | DATE_YYYY_MM_DD | FILE_UPLOAD | TEXT_OR...,FILE_UPLOAD



Candidate: data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/data/reference/follow_up_question_selection_rules_v1.csv
Shape: (7, 6)
Columns: ['selection_rule_id', 'rule_name', 'priority', 'condition', 'agent_action', 'rationale']


,selection_rule_id,rule_name,priority,condition,agent_action,rationale
0,FSEL-001,Higher-precedence outcome blocks follow-up,10,Any applicable rule has already produced MANUA...,Do not ask follow-up questions. Return the exi...,The agent must not burden the customer with do...
1,FSEL-002,Follow-up only for INFO_REQUIRED,20,An applicable deterministic rule produces INFO...,Select only catalogue questions mapped to the ...,Questions must be rule-grounded and not improv...
2,FSEL-003,Eligibility facts before evidence,30,Both eligibility facts and required evidence a...,Ask eligibility-fact questions first. Reassess...,The system should not request evidence for a c...
3,FSEL-004,Maximum questions per interaction,40,More than one valid catalogue question is sele...,Ask a maximum of three questions in one messag...,Reduces customer burden and keeps the interact...
4,FSEL-005,No duplicate question,50,The same response field has already been reque...,Use the same question at most once. If still u...,Prevents looping and supports bounded agent be...
5,FSEL-006,No customer resolution of internal conflicts,60,"The issue is a source-system conflict, duplica...",Do not generate a customer question. Route to ...,Customers must not be asked to solve internal ...
6,FSEL-007,Neutral communication,70,A follow-up message is generated.,Use the catalogue wording with a neutral expla...,Matches the operational safety and communicati...



Candidate: data/staging/BYOC_DeviceProtect_Claims_Triage_Dataset_v1/docs/follow_up_question_tool_contract_v1.json
Top-level JSON type: dict
Top-level keys: ['tool_name', 'version', 'purpose', 'input', 'output', 'constraints']
JSON preview:
{
  "tool_name": "select_follow_up_questions",
  "version": "1.0",
  "purpose": "Select controlled customer follow-up questions after deterministic triage produces INFO_REQUIRED.",
  "input": {
    "claim_id": "string",
    "current_disposition": "PROCEED | INFO_REQUIRED | MANUAL_REVIEW | NOT_ELIGIBLE",
    "unmet_rule_ids": [
      "string"
    ],
    "claim_category": "string",
    "evidence_profile_id": "string or null",
    "evidence_status_summary": [
      {
        "evidence_type": "string",
        "document_status": "string"
      }
    ],
    "questions_already_asked": [
      "string"
    ]
  },
  "output": {
    "follow_up_required": "boolean",
    "question_ids": [
      "string"
    ],
    "customer_message": "string",
    "deferred_re

In [4]:
# ============================================================
# Step 1D: Inspect Follow-Up Communication SOP
# ============================================================

follow_up_sop_files = [
    PROJECT_ROOT / relative_path
    for relative_path in filename_candidates["relative_path"].tolist()
    if "knowledge_base" in relative_path
    and "follow" in Path(relative_path).name.casefold()
]

for sop_path in follow_up_sop_files:
    print("\n" + "=" * 80)
    print("SOP:", sop_path.relative_to(PROJECT_ROOT))
    print("=" * 80)
    print(sop_path.read_text(encoding="utf-8", errors="ignore"))


SOP: data/knowledge_base/KB-003_follow_up_communication_guide_v1.md
# Follow-up Communication Guide

**Document ID:** KB-003  
**Version:** 1.0  
**Effective date:** 2026-06-23  
**Purpose:** Define safe, concise, and neutral communication when more information is required for claims triage.

## 1. Communication objective

A follow-up message should obtain the minimum information needed to continue assessment. It should not accuse, pressure, promise an outcome, or ask the customer to interpret policy rules.

## 2. Required style

Use language that is:

- Specific: identify the missing fact or document.
- Neutral: state what is needed without implying fault.
- Actionable: explain how the customer can provide it.
- Bounded: ask only for information needed for the current decision step.
- Non-committal: do not state or imply final approval, final rejection, payment amount, repair authorization, or fraud concern.

## 3. Approved response structure

1. Acknowledge the claim submission.
2. 

## Step 2 — Extend the Runtime Data Loader

The approved follow-up catalog and selection rules exist in the runtime reference folder but are not yet exposed by `load_runtime_data()`.

This step inspects the current loader implementation before adding only the required datasets. The staging copies are intentionally excluded.

In [5]:
# ============================================================
# Step 2: Inspect Current Runtime Data Loader
# ============================================================

import inspect

import src.data_loader as data_loader

print(inspect.getsource(data_loader))

from pathlib import Path

import pandas as pd


def get_project_root() -> Path:
    """Return the project root directory."""
    return Path(__file__).resolve().parents[1]


def load_runtime_data() -> dict[str, pd.DataFrame]:
    """
    Load approved runtime datasets used by the deterministic triage workflow.

    Held-out evaluation assets are intentionally excluded.
    """
    root = get_project_root()
    reference_dir = root / "data" / "runtime" / "reference"
    claims_dir = root / "data" / "runtime" / "claims"

    return {
        "plan_master": pd.read_csv(
            reference_dir / "protection_plan_master_v1_1.csv"
        ),
        "coverage_matrix": pd.read_csv(
            reference_dir / "plan_coverage_matrix_v1.csv"
        ),
        "evidence_requirements": pd.read_csv(
            reference_dir / "evidence_profile_requirements_v1.csv"
        ),
        "policy_rules": pd.read_csv(
            reference_dir / "policy_rule_catalog_v1_2.csv"
        ),
        "poli

In [6]:
# ============================================================
# Step 2B: Validate Follow-Up Runtime Data Loading
# ============================================================

import importlib

import src.data_loader as data_loader

importlib.reload(data_loader)

data = data_loader.load_runtime_data()

follow_up_catalog = data["follow_up_question_catalog"]
selection_rules = data["follow_up_question_selection_rules"]

print("Follow-up catalogue rows:", len(follow_up_catalog))
print("Selection-rule rows:", len(selection_rules))

print("\nCatalogue columns:")
print(follow_up_catalog.columns.tolist())

print("\nSelection-rule columns:")
print(selection_rules.columns.tolist())

assert len(follow_up_catalog) == 14
assert len(selection_rules) == 7
assert (follow_up_catalog["active_flag"] == "Yes").all()

print("\nValidation passed: approved follow-up assets are available at runtime.")

Follow-up catalogue rows: 14
Selection-rule rows: 7

Catalogue columns:
['question_id', 'question_name', 'trigger_type', 'trigger_condition', 'source_rule_id', 'decision_stage', 'applicable_claim_categories', 'evidence_profile_id', 'response_field', 'response_format', 'question_priority', 'customer_facing_question', 'plain_language_reason', 'agent_handling_after_response', 'max_repeats', 'active_flag']

Selection-rule columns:
['selection_rule_id', 'rule_name', 'priority', 'condition', 'agent_action', 'rationale']

Validation passed: approved follow-up assets are available at runtime.


## Step 3 — Review Deterministic Question Mappings

This step reviews the complete approved follow-up catalogue alongside the selection rules.

The purpose is to confirm how each question is grounded in a deterministic source rule, claim category, evidence profile, priority, and repeat limit before implementing the selector.

In [7]:
# ============================================================
# Step 3: Review Follow-Up Question Mappings
# ============================================================

catalogue_review_columns = [
    "question_id",
    "question_name",
    "source_rule_id",
    "decision_stage",
    "trigger_type",
    "applicable_claim_categories",
    "evidence_profile_id",
    "response_field",
    "response_format",
    "question_priority",
    "max_repeats",
    "active_flag",
    "customer_facing_question",
]

catalogue_review = (
    follow_up_catalog[catalogue_review_columns]
    .sort_values(
        by=[
            "source_rule_id",
            "question_priority",
            "question_id",
        ]
    )
    .reset_index(drop=True)
)

selection_rules_review = (
    selection_rules
    .sort_values("priority")
    .reset_index(drop=True)
)

print("Approved follow-up question catalogue:")
display(catalogue_review)

print("\nSelection rules:")
display(selection_rules_review)

print("\nQuestion count by deterministic source rule:")
display(
    catalogue_review.groupby(
        ["source_rule_id", "decision_stage"],
        dropna=False,
    )
    .size()
    .reset_index(name="question_count")
    .sort_values(["source_rule_id", "decision_stage"])
    .reset_index(drop=True)
)

assert len(catalogue_review) == 14
assert catalogue_review["question_id"].is_unique
assert catalogue_review["question_priority"].notna().all()

print(
    "\nValidation passed: all approved follow-up questions have unique "
    "identifiers and deterministic rule mappings."
)

Approved follow-up question catalogue:


,question_id,question_name,source_rule_id,decision_stage,trigger_type,applicable_claim_categories,evidence_profile_id,response_field,response_format,question_priority,max_repeats,active_flag,customer_facing_question
0,FUP-004,Clarify incident category,CLM-001,ELIGIBILITY_FACTS,MISSING_OR_AMBIGUOUS_FIELD,UNSPECIFIED,NaN,claim_category_selected,ENUM_OR_FREE_TEXT,40,1,Yes,Please describe what happened to the device an...
1,FUP-001,Confirm policy or customer reference,DATA-001,ELIGIBILITY_FACTS,MISSING_FIELD,ALL,NaN,policy_or_customer_reference,TEXT,10,1,Yes,Please provide your policy reference or custom...
2,FUP-003,Provide claimed device identifier,DEV-001,ELIGIBILITY_FACTS,MISSING_OR_INVALID_FIELD,ALL,NaN,claimed_device_identifier,TEXT,30,1,Yes,Please provide the IMEI or serial number for t...
3,FUP-002,Confirm incident date,ELG-001,ELIGIBILITY_FACTS,MISSING_OR_INVALID_FIELD,ALL,NaN,reported_incident_date,DATE_YYYY_MM_DD,20,1,Yes,Please confirm the date on which the incident ...
4,FUP-005,Upload screen-damage photo,EVD-001,REQUIRED_EVIDENCE,MISSING_REQUIRED_EVIDENCE,SCREEN_DAMAGE,EVD-SCREEN-01,DAMAGE_PHOTO,FILE_UPLOAD,50,1,Yes,Please upload a clear photo showing the report...
5,FUP-006,Re-upload readable screen-damage photo,EVD-001,REQUIRED_EVIDENCE,UNREADABLE_OR_INVALID_EVIDENCE,SCREEN_DAMAGE,EVD-SCREEN-01,DAMAGE_PHOTO,FILE_UPLOAD,50,1,Yes,The uploaded image could not be read clearly. ...
6,FUP-007,Upload accidental-damage photo,EVD-001,REQUIRED_EVIDENCE,MISSING_REQUIRED_EVIDENCE,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,DAMAGE_PHOTO,FILE_UPLOAD,50,1,Yes,Please upload a clear photo showing the report...
7,FUP-008,Re-upload readable accidental-damage photo,EVD-001,REQUIRED_EVIDENCE,UNREADABLE_OR_INVALID_EVIDENCE,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,DAMAGE_PHOTO,FILE_UPLOAD,50,1,Yes,The uploaded image could not be read clearly. ...
8,FUP-009,Upload liquid-damage photo,EVD-001,REQUIRED_EVIDENCE,MISSING_REQUIRED_EVIDENCE,LIQUID_DAMAGE,EVD-LIQUID-01,DAMAGE_PHOTO,FILE_UPLOAD,50,1,Yes,Please upload a clear photo showing the report...
9,FUP-010,Re-upload readable liquid-damage photo,EVD-001,REQUIRED_EVIDENCE,UNREADABLE_OR_INVALID_EVIDENCE,LIQUID_DAMAGE,EVD-LIQUID-01,DAMAGE_PHOTO,FILE_UPLOAD,50,1,Yes,The uploaded image could not be read clearly. ...



Selection rules:


,selection_rule_id,rule_name,priority,condition,agent_action,rationale
0,FSEL-001,Higher-precedence outcome blocks follow-up,10,Any applicable rule has already produced MANUA...,Do not ask follow-up questions. Return the exi...,The agent must not burden the customer with do...
1,FSEL-002,Follow-up only for INFO_REQUIRED,20,An applicable deterministic rule produces INFO...,Select only catalogue questions mapped to the ...,Questions must be rule-grounded and not improv...
2,FSEL-003,Eligibility facts before evidence,30,Both eligibility facts and required evidence a...,Ask eligibility-fact questions first. Reassess...,The system should not request evidence for a c...
3,FSEL-004,Maximum questions per interaction,40,More than one valid catalogue question is sele...,Ask a maximum of three questions in one messag...,Reduces customer burden and keeps the interact...
4,FSEL-005,No duplicate question,50,The same response field has already been reque...,Use the same question at most once. If still u...,Prevents looping and supports bounded agent be...
5,FSEL-006,No customer resolution of internal conflicts,60,"The issue is a source-system conflict, duplica...",Do not generate a customer question. Route to ...,Customers must not be asked to solve internal ...
6,FSEL-007,Neutral communication,70,A follow-up message is generated.,Use the catalogue wording with a neutral expla...,Matches the operational safety and communicati...



Question count by deterministic source rule:


,source_rule_id,decision_stage,question_count
0,CLM-001,ELIGIBILITY_FACTS,1
1,DATA-001,ELIGIBILITY_FACTS,1
2,DEV-001,ELIGIBILITY_FACTS,1
3,ELG-001,ELIGIBILITY_FACTS,1
4,EVD-001,REQUIRED_EVIDENCE,10



Validation passed: all approved follow-up questions have unique identifiers and deterministic rule mappings.


In [8]:
# ============================================================
# Step 3B: Inspect Available Claim-Context Fields
# ============================================================

import inspect

import src.claim_context as claim_context

print(inspect.getsource(claim_context))

from __future__ import annotations

import pandas as pd

from src.tools.claims_history import (
    calculate_claim_usage,
    get_policy_claim_history,
)
from src.tools.coverage_lookup import (
    classify_coverage_result,
    get_coverage_record,
)
from src.tools.evidence_assessment import assess_evidence
from src.tools.evidence_lookup import get_evidence_documents
from src.tools.plan_configuration import assess_plan_configuration
from src.tools.policy_eligibility import assess_policy_for_incident
from src.tools.policy_lookup import (
    classify_lookup_result,
    find_policy_records,
)
from src.tools.risk_lookup import (
    get_claim_risk_results,
    summarise_risk_results,
)


def assemble_claim_context(
    data: dict[str, pd.DataFrame],
    claim_id: str,
) -> dict:
    """
    Assemble structured facts for one claim.

    This function does not make a final triage decision.
    """
    claims = data["development_claims"]

    claim_records = claims[claims["claim_id"] == cla

## Step 3C — Inspect INFO_REQUIRED Claim Contexts

This step examines representative development claims for each deterministic `INFO_REQUIRED` rule.

The objective is to map the existing claim context and evidence-assessment fields to the approved follow-up catalogue without relying on assumed field names or conditions.

In [9]:
# ============================================================
# Step 3C: Inspect INFO_REQUIRED Claim Contexts
# ============================================================

from pprint import pprint

from src.claim_context import assemble_claim_context
from src.triage_engine import triage_claim


INFO_REQUIRED_RULE_IDS = (
    "DATA-001",
    "ELG-001",
    "DEV-001",
    "CLM-001",
    "EVD-001",
)

development_claims = data["development_claims"].copy()
representative_info_required_claims = {}

for claim_id in development_claims["claim_id"].astype(str):
    context = assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    if (
        decision["triage_outcome"] == "INFO_REQUIRED"
        and decision["triggering_rule_id"] in INFO_REQUIRED_RULE_IDS
        and decision["triggering_rule_id"]
        not in representative_info_required_claims
    ):
        representative_info_required_claims[
            decision["triggering_rule_id"]
        ] = claim_id

print("Representative development claims by triggering rule:")
pprint(representative_info_required_claims)

for rule_id in INFO_REQUIRED_RULE_IDS:
    claim_id = representative_info_required_claims.get(rule_id)

    if claim_id is None:
        print("\n" + "=" * 80)
        print(f"{rule_id}: no development example available.")
        continue

    context = assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    print("\n" + "=" * 80)
    print(f"Rule: {rule_id} | Claim: {claim_id}")
    print("=" * 80)

    print("\nDeterministic decision:")
    pprint(
        {
            "triage_outcome": decision["triage_outcome"],
            "triggering_rule_id": decision["triggering_rule_id"],
            "decision_reason": decision["decision_reason"],
        }
    )

    print("\nRelevant context:")
    pprint(
        {
            "claim": context["claim"],
            "policy_lookup_status": context["policy_lookup_status"],
            "policy_eligibility": context["policy_eligibility"],
            "device_match": context["device_match"],
            "coverage": context["coverage"],
            "evidence": context["evidence"],
        }
    )

Representative development claims by triggering rule:
{'CLM-001': 'CLM-000086',
 'DEV-001': 'CLM-000098',
 'ELG-001': 'CLM-000071',
 'EVD-001': 'CLM-000117'}

DATA-001: no development example available.

Rule: ELG-001 | Claim: CLM-000071

Deterministic decision:
{'decision_reason': 'A valid incident date is required before incident-date '
                    'eligibility can be assessed.',
 'triage_outcome': 'INFO_REQUIRED',
 'triggering_rule_id': 'ELG-001'}

Relevant context:
{'claim': {'claim_category_selected': 'MECHANICAL_BREAKDOWN',
           'claim_id': 'CLM-000071',
           'claimed_device_identifier': 'DVC-00000045',
           'customer_id_provided': 'CUS-0062',
           'evidence_bundle_id': 'EBND-000071',
           'policy_id_provided': 'POL-00045',
           'reported_incident_date': nan,
           'requested_service_type': 'UNSPECIFIED'},
 'coverage': {'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD',
              'covered_flag': True,
              'evidence_p

In [10]:
# ============================================================
# Step 3D: Inspect All Development EVD-001 Evidence Conditions
# ============================================================

evd001_rows = []

for claim_id in development_claims["claim_id"].astype(str):
    context = assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    if decision["triggering_rule_id"] != "EVD-001":
        continue

    evidence = context["evidence"]
    claim = context["claim"]

    evd001_rows.append(
        {
            "claim_id": claim_id,
            "claim_category": claim["claim_category_selected"],
            "evidence_profile_id": evidence["evidence_profile_id"],
            "evidence_status": evidence["evidence_assessment_status"],
            "missing_evidence": evidence[
                "missing_required_evidence_types"
            ],
            "unreadable_evidence": evidence[
                "unreadable_required_evidence_types"
            ],
            "decision_reason": decision["decision_reason"],
        }
    )

evd001_review = (
    pd.DataFrame(evd001_rows)
    .sort_values(
        by=[
            "claim_category",
            "claim_id",
        ]
    )
    .reset_index(drop=True)
)

print("Development claims routed by EVD-001:", len(evd001_review))
display(evd001_review)

print("\nEvidence-condition counts:")
display(
    evd001_review.assign(
        evidence_condition=evd001_review.apply(
            lambda row: (
                "MISSING"
                if row["missing_evidence"]
                else "UNREADABLE"
                if row["unreadable_evidence"]
                else "OTHER"
            ),
            axis=1,
        )
    )
    .groupby(
        ["claim_category", "evidence_condition"],
        dropna=False,
    )
    .size()
    .reset_index(name="claim_count")
    .sort_values(
        ["claim_category", "evidence_condition"]
    )
    .reset_index(drop=True)
)

Development claims routed by EVD-001: 11


,claim_id,claim_category,evidence_profile_id,evidence_status,missing_evidence,unreadable_evidence,decision_reason
0,CLM-000125,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,INCOMPLETE,[],[DAMAGE_PHOTO],Required evidence is missing or unreadable. Re...
1,CLM-000127,ACCIDENTAL_DAMAGE,EVD-ACCIDENTAL-01,INCOMPLETE,[DAMAGE_PHOTO],[],Required evidence is missing or unreadable. Re...
2,CLM-000117,LIQUID_DAMAGE,EVD-LIQUID-01,INCOMPLETE,[DAMAGE_PHOTO],[],Required evidence is missing or unreadable. Re...
3,CLM-000119,LIQUID_DAMAGE,EVD-LIQUID-01,INCOMPLETE,[DAMAGE_PHOTO],[],Required evidence is missing or unreadable. Re...
4,CLM-000126,LIQUID_DAMAGE,EVD-LIQUID-01,INCOMPLETE,[],[DAMAGE_PHOTO],Required evidence is missing or unreadable. Re...
5,CLM-000118,MECHANICAL_BREAKDOWN,EVD-MECHANICAL-01,INCOMPLETE,[DIAGNOSTIC_REPORT],[],Required evidence is missing or unreadable. Re...
6,CLM-000122,MECHANICAL_BREAKDOWN,EVD-MECHANICAL-01,INCOMPLETE,[DIAGNOSTIC_REPORT],[],Required evidence is missing or unreadable. Re...
7,CLM-000124,MECHANICAL_BREAKDOWN,EVD-MECHANICAL-01,INCOMPLETE,[DIAGNOSTIC_REPORT],[],Required evidence is missing or unreadable. Re...
8,CLM-000120,SCREEN_DAMAGE,EVD-SCREEN-01,INCOMPLETE,[],[DAMAGE_PHOTO],Required evidence is missing or unreadable. Re...
9,CLM-000121,SCREEN_DAMAGE,EVD-SCREEN-01,INCOMPLETE,[DAMAGE_PHOTO],[],Required evidence is missing or unreadable. Re...



Evidence-condition counts:


,claim_category,evidence_condition,claim_count
0,ACCIDENTAL_DAMAGE,MISSING,1
1,ACCIDENTAL_DAMAGE,UNREADABLE,1
2,LIQUID_DAMAGE,MISSING,2
3,LIQUID_DAMAGE,UNREADABLE,1
4,MECHANICAL_BREAKDOWN,MISSING,3
5,SCREEN_DAMAGE,MISSING,2
6,SCREEN_DAMAGE,UNREADABLE,1


In [11]:
follow_up_sop_path = (
    PROJECT_ROOT
    / "data"
    / "knowledge_base"
    / "KB-003_follow_up_communication_guide_v1.md"
)

print(follow_up_sop_path.read_text(encoding="utf-8"))

# Follow-up Communication Guide

**Document ID:** KB-003  
**Version:** 1.0  
**Effective date:** 2026-06-23  
**Purpose:** Define safe, concise, and neutral communication when more information is required for claims triage.

## 1. Communication objective

A follow-up message should obtain the minimum information needed to continue assessment. It should not accuse, pressure, promise an outcome, or ask the customer to interpret policy rules.

## 2. Required style

Use language that is:

- Specific: identify the missing fact or document.
- Neutral: state what is needed without implying fault.
- Actionable: explain how the customer can provide it.
- Bounded: ask only for information needed for the current decision step.
- Non-committal: do not state or imply final approval, final rejection, payment amount, repair authorization, or fraud concern.

## 3. Approved response structure

1. Acknowledge the claim submission.
2. State the missing or unclear item.
3. Explain why it is needed in pla

## Step 4B — Validate Controlled Question Selection

This step checks that each representative `INFO_REQUIRED` claim receives the matching approved catalogue question. No LLM is used.

In [12]:
# ============================================================
# Step 4B: Controlled Follow-Up Selection Validation
# ============================================================

from src.tools.follow_up_questions import (
    select_follow_up_questions,
)

expected_question_by_claim = {
    "CLM-000071": "FUP-002",  # Missing incident date
    "CLM-000098": "FUP-003",  # Missing device identifier
    "CLM-000086": "FUP-004",  # Unspecified category
    "CLM-000117": "FUP-009",  # Missing liquid-damage photo
    "CLM-000125": "FUP-008",  # Unreadable accidental-damage photo
}

selection_validation_rows = []

for claim_id, expected_question_id in expected_question_by_claim.items():
    context = assemble_claim_context(
        data=data,
        claim_id=claim_id,
    )

    decision = triage_claim(context)

    selection_result = select_follow_up_questions(
        context=context,
        deterministic_decision=decision,
        question_catalog=data["follow_up_question_catalog"],
    )

    selection_validation_rows.append(
        {
            "claim_id": claim_id,
            "triggering_rule_id": decision["triggering_rule_id"],
            "expected_question_id": expected_question_id,
            "selected_question_ids": selection_result["question_ids"],
            "selection_status": selection_result["selection_status"],
            "follow_up_required": selection_result["follow_up_required"],
            "customer_message": selection_result["customer_message"],
        }
    )

selection_validation = pd.DataFrame(selection_validation_rows)

display(selection_validation)

assert (
    selection_validation["follow_up_required"]
).all()

assert (
    selection_validation["selection_status"]
    == "QUESTIONS_SELECTED"
).all()

for row in selection_validation.itertuples(index=False):
    assert row.selected_question_ids == [row.expected_question_id]

print(
    "\nValidation passed: each INFO_REQUIRED case received the expected "
    "approved catalogue question."
)

,claim_id,triggering_rule_id,expected_question_id,selected_question_ids,selection_status,follow_up_required,customer_message
0,CLM-000071,ELG-001,FUP-002,[FUP-002],QUESTIONS_SELECTED,True,Thanks for submitting your claim. Please confi...
1,CLM-000098,DEV-001,FUP-003,[FUP-003],QUESTIONS_SELECTED,True,Thanks for submitting your claim. Please provi...
2,CLM-000086,CLM-001,FUP-004,[FUP-004],QUESTIONS_SELECTED,True,Thanks for submitting your claim. Please descr...
3,CLM-000117,EVD-001,FUP-009,[FUP-009],QUESTIONS_SELECTED,True,Thanks for submitting your claim. Please uploa...
4,CLM-000125,EVD-001,FUP-008,[FUP-008],QUESTIONS_SELECTED,True,Thanks for submitting your claim. The uploaded...



Validation passed: each INFO_REQUIRED case received the expected approved catalogue question.


## Step 5 — Prepare Follow-Up Tool Integration

The follow-up selector is validated independently. This step inspects the existing deterministic triage adapter before integrating controlled follow-up selection into the protected workflow.

The objective is to reuse the authoritative deterministic decision and avoid reimplementing triage logic.

In [13]:
# ============================================================
# Step 5: Inspect Deterministic Triage Adapter
# ============================================================

import inspect

import src.tools.deterministic_triage as deterministic_triage

print(inspect.getsource(deterministic_triage))

from __future__ import annotations

from typing import Any, Mapping

from src.claim_context import assemble_claim_context
from src.triage_engine import triage_claim


TOOL_NAME = "deterministic_triage"
TOOL_VERSION = "rules_baseline_v1"

AUTHORITATIVE_FIELDS = [
    "triage_outcome",
    "triggering_rule_id",
    "precedence_stage",
    "decision_reason",
    "rule_trace",
    "system_limitations",
]


def run_deterministic_triage(
    data: Mapping[str, Any],
    claim_id: str,
) -> dict[str, Any]:
    """
    Assemble claim context and return the authoritative deterministic triage.

    This adapter does not load evaluation data, infer new facts, approve
    payment, determine fraud, or override the rules-engine decision.
    """
    if not isinstance(claim_id, str) or not claim_id.strip():
        raise ValueError("claim_id must be a non-empty string.")

    normalised_claim_id = claim_id.strip()

    context = assemble_claim_context(
        data=data,
        claim_id=normalised_c

## Step 6B — Validate the Follow-Up Selection Adapter

This step validates the adapter that connects the authoritative deterministic triage result to controlled catalogue-based follow-up selection.

The adapter does not rerun or modify the triage decision.

In [14]:
# ============================================================
# Step 6B: Follow-Up Selection Adapter Validation
# ============================================================

from pprint import pprint

from src.tools.deterministic_triage import run_deterministic_triage
from src.tools.follow_up_selection import (
    run_controlled_follow_up_selection,
)


info_required_tool_result = run_deterministic_triage(
    data=data,
    claim_id="CLM-000117",
)

info_required_follow_up = run_controlled_follow_up_selection(
    data=data,
    claim_id="CLM-000117",
    deterministic_tool_result=info_required_tool_result,
)

proceed_tool_result = run_deterministic_triage(
    data=data,
    claim_id="CLM-000001",
)

proceed_follow_up = run_controlled_follow_up_selection(
    data=data,
    claim_id="CLM-000001",
    deterministic_tool_result=proceed_tool_result,
)

print("INFO_REQUIRED follow-up selection:")
pprint(info_required_follow_up)

print("\nPROCEED follow-up selection:")
pprint(proceed_follow_up)

assert (
    info_required_tool_result["deterministic_decision"]["triage_outcome"]
    == "INFO_REQUIRED"
)

assert (
    info_required_follow_up["follow_up_selection"]["question_ids"]
    == ["FUP-009"]
)

assert (
    proceed_tool_result["deterministic_decision"]["triage_outcome"]
    == "PROCEED"
)

assert (
    proceed_follow_up["follow_up_selection"]["follow_up_required"]
    is False
)

assert (
    proceed_follow_up["follow_up_selection"]["question_ids"]
    == []
)

print(
    "\nValidation passed: controlled questions were selected only for "
    "INFO_REQUIRED without changing deterministic triage routing."
)

INFO_REQUIRED follow-up selection:
{'authority_notice': 'Follow-up selection is deterministic and uses only '
                     'approved catalogue questions. It does not alter the '
                     'authoritative triage outcome, triggering rule, or '
                     'decision rationale.',
 'claim_id': 'CLM-000117',
 'decision_source': 'deterministic_triage:rules_baseline_v1',
 'follow_up_selection': {'claim_id': 'CLM-000117',
                         'customer_message': 'Thanks for submitting your '
                                             'claim. Please upload a clear '
                                             'photo showing the reported '
                                             'liquid exposure effects or '
                                             'device damage. The image helps '
                                             'us assess the reported '
                                             'liquid-damage symptoms. Once we '
                        

## Step 7B — Reference Workflow Follow-Up Integration

This step confirms that controlled follow-up selection is included in the protected workflow output without changing deterministic triage authority or passing customer-question generation to the LLM.

In [15]:
# ============================================================
# Step 7B: Reference Workflow Follow-Up Integration
# ============================================================

import importlib
from pprint import pprint

import src.agent.orchestrator as orchestrator

importlib.reload(orchestrator)

run_guarded_claim_orchestrator = (
    orchestrator.run_guarded_claim_orchestrator
)

info_required_workflow = run_guarded_claim_orchestrator(
    data=data,
    claim_id="CLM-000117",
)

proceed_workflow = run_guarded_claim_orchestrator(
    data=data,
    claim_id="CLM-000001",
)

print("INFO_REQUIRED controlled follow-up:")
pprint(
    info_required_workflow["final_response"][
        "controlled_follow_up"
    ]
)

print("\nPROCEED controlled follow-up:")
pprint(
    proceed_workflow["final_response"][
        "controlled_follow_up"
    ]
)

print("\nINFO_REQUIRED workflow trace:")
pprint(info_required_workflow["workflow_trace"])

assert info_required_workflow["workflow_version"] == "reference_v3"

assert [
    item["node"]
    for item in info_required_workflow["workflow_trace"]
] == [
    "deterministic_triage",
    "controlled_follow_up_selection",
    "explanation_proposal",
    "agent_content_safety_guardrail",
    "response_guardrail",
]

assert (
    info_required_workflow["final_response"]["triage_outcome"]
    == "INFO_REQUIRED"
)

assert (
    info_required_workflow["final_response"][
        "controlled_follow_up"
    ]["question_ids"]
    == ["FUP-009"]
)

assert (
    proceed_workflow["final_response"]["triage_outcome"]
    == "PROCEED"
)

assert (
    proceed_workflow["final_response"][
        "controlled_follow_up"
    ]["follow_up_required"]
    is False
)

assert (
    proceed_workflow["final_response"][
        "controlled_follow_up"
    ]["question_ids"]
    == []
)

print(
    "\nValidation passed: follow-up selection is deterministic, "
    "catalogue-based, and separate from triage authority."
)

INFO_REQUIRED controlled follow-up:
{'claim_id': 'CLM-000117',
 'customer_message': 'Thanks for submitting your claim. Please upload a clear '
                     'photo showing the reported liquid exposure effects or '
                     'device damage. The image helps us assess the reported '
                     'liquid-damage symptoms. Once we receive the requested '
                     'information, the claim assessment can continue.',
 'deferred_requirements': [],
 'follow_up_required': True,
 'question_ids': ['FUP-009'],
 'selected_questions': [{'agent_handling_after_response': 'Check for a '
                                                          'readable, relevant '
                                                          'photo. A repair '
                                                          'quote may be '
                                                          'considered if '
                                                          'supplied but is '
      

In [16]:
# ============================================================
# Step 9: LangGraph Controlled Follow-Up Integration
# ============================================================

import importlib
from pprint import pprint

import src.agent.langgraph_orchestrator as langgraph_orchestrator

importlib.reload(langgraph_orchestrator)

run_langgraph_guarded_claim_triage = (
    langgraph_orchestrator.run_langgraph_guarded_claim_triage
)

langgraph_info_required = run_langgraph_guarded_claim_triage(
    data=data,
    claim_id="CLM-000117",
)

langgraph_proceed = run_langgraph_guarded_claim_triage(
    data=data,
    claim_id="CLM-000001",
)

print("INFO_REQUIRED follow-up:")
pprint(
    langgraph_info_required["final_response"][
        "controlled_follow_up"
    ]
)

print("\nPROCEED follow-up:")
pprint(
    langgraph_proceed["final_response"][
        "controlled_follow_up"
    ]
)

print("\nINFO_REQUIRED workflow trace:")
pprint(langgraph_info_required["workflow_trace"])

assert langgraph_info_required["workflow_version"] == "langgraph_v3"

assert [
    item["node"]
    for item in langgraph_info_required["workflow_trace"]
] == [
    "deterministic_triage",
    "controlled_follow_up_selection",
    "explanation_proposal",
    "agent_content_safety_guardrail",
    "response_guardrail",
]

assert (
    langgraph_info_required["final_response"]["triage_outcome"]
    == "INFO_REQUIRED"
)

assert (
    langgraph_info_required["final_response"][
        "controlled_follow_up"
    ]["question_ids"]
    == ["FUP-009"]
)

assert (
    langgraph_proceed["final_response"]["triage_outcome"]
    == "PROCEED"
)

assert (
    langgraph_proceed["final_response"][
        "controlled_follow_up"
    ]["follow_up_required"]
    is False
)

print(
    "\nValidation passed: LangGraph returned controlled follow-up "
    "output without changing deterministic triage authority."
)

INFO_REQUIRED follow-up:
{'claim_id': 'CLM-000117',
 'customer_message': 'Thanks for submitting your claim. Please upload a clear '
                     'photo showing the reported liquid exposure effects or '
                     'device damage. The image helps us assess the reported '
                     'liquid-damage symptoms. Once we receive the requested '
                     'information, the claim assessment can continue.',
 'deferred_requirements': [],
 'follow_up_required': True,
 'question_ids': ['FUP-009'],
 'selected_questions': [{'agent_handling_after_response': 'Check for a '
                                                          'readable, relevant '
                                                          'photo. A repair '
                                                          'quote may be '
                                                          'considered if '
                                                          'supplied but is '
                 

## Step 10 — Live OpenAI Workflow with Controlled Follow-Up

This step runs an `INFO_REQUIRED` development claim through the complete protected LangGraph workflow.

The OpenAI component produces analyst-facing explanation content only. The customer follow-up message remains deterministic and is selected solely from the approved question catalogue.

In [17]:
# ============================================================
# Step 10: Live OpenAI Workflow with Controlled Follow-Up
# ============================================================

from pprint import pprint

from src.agent.openai_explainer import (
    build_openai_explanation_proposal,
)

live_info_required_workflow = run_langgraph_guarded_claim_triage(
    data=data,
    claim_id="CLM-000117",
    proposal_builder=build_openai_explanation_proposal,
)

final_response = live_info_required_workflow["final_response"]

print("Workflow trace:")
pprint(live_info_required_workflow["workflow_trace"])

print("\nAnalyst-facing explanation content:")
pprint(final_response["agent_content"])

print("\nDeterministic controlled follow-up:")
pprint(final_response["controlled_follow_up"])

print("\nAuthority guardrail:")
pprint(final_response["authority_guardrail"])

assert final_response["triage_outcome"] == "INFO_REQUIRED"
assert final_response["triggering_rule_id"] == "EVD-001"

assert (
    final_response["controlled_follow_up"]["follow_up_required"]
    is True
)

assert (
    final_response["controlled_follow_up"]["question_ids"]
    == ["FUP-009"]
)

assert (
    final_response["authority_guardrail"]["status"]
    == "ALIGNED"
)

assert (
    live_info_required_workflow["content_safety"][
        "content_safety_status"
    ]
    in {"SAFE", "FALLBACK_APPLIED"}
)

print(
    "\nValidation passed: the LLM explained the deterministic routing, "
    "while the approved catalogue provided the customer follow-up."
)

Workflow trace:
[{'authoritative': True,
  'node': 'deterministic_triage',
  'status': 'COMPLETED',
  'tool_version': 'rules_baseline_v1'},
 {'follow_up_required': True,
  'node': 'controlled_follow_up_selection',
  'question_ids': ['FUP-009'],
  'selection_status': 'QUESTIONS_SELECTED',
  'status': 'COMPLETED'},
 {'node': 'explanation_proposal',
  'proposal_source': 'CUSTOM',
  'proposed_field_names': ['case_summary',
                           'next_step_message',
                           'reviewer_note'],
  'status': 'COMPLETED'},
 {'content_safety_status': 'SAFE',
  'content_safety_violations': [],
  'fallback_used': False,
  'node': 'agent_content_safety_guardrail',
  'status': 'COMPLETED'},
 {'guardrail_status': 'ALIGNED',
  'node': 'response_guardrail',
  'status': 'COMPLETED'}]

Analyst-facing explanation content:
{'case_summary': 'Claim CLM-000117 routed to INFO_REQUIRED. The triggering '
                 'rule was EVD-001 at precedence stage 5 because the evidence '
       